## Phase 1.1 – Feature ablation for the late delivery model

Additional information available at order time is added to the Phase 0 feature set one group at a time: item price, quantity and discount; store coordinates; and high-cardinality location and product columns. High-cardinality columns are target-encoded (each category is replaced by its smoothed historical late rate, learned on the training data with internal cross-fitting).

The scheduled shipping days column is not used, because it is fully determined by the shipping mode (First Class = 1, Same Day = 0, Second Class = 2, Standard Class = 4 days).

First, all variants are compared on a single time-based split (train until 2017-03, test 2017-04 – 2017-09). The most promising variant is then compared with the base model using rolling-origin validation and a cluster bootstrap confidence interval.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier

from src.data import load_data
from src.features import (DELAY_NUMERIC, DELAY_CATEGORICAL, DELAY_TARGET,
                          DELAY_FEATURES, DELAY_FEATURES_V2,
                          delay_preprocessor, delay_preprocessor_v2)
from src.evaluation import rolling_origin_months, cluster_bootstrap

data = load_data()
data["period"] = data["order_date"].dt.to_period("M")
data = data[data["period"] <= pd.Period("2017-09", "M")].reset_index(drop=True)
print(data.shape)

(171962, 52)


In [2]:
train = data[data["period"] < pd.Period("2017-04", "M")]
test = data[data["period"] >= pd.Period("2017-04", "M")]

def xgb():
    return XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                         random_state=42, n_jobs=-1, eval_metric="logloss")

def try_features(name, numeric, one_hot, target_enc=()):
    parts = [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), list(one_hot))]
    if target_enc:
        parts.append(("te", TargetEncoder(random_state=42), list(target_enc)))
    parts.append(("num", "passthrough", list(numeric)))
    model = Pipeline([("prep", ColumnTransformer(parts)), ("model", xgb())])
    cols = list(numeric) + list(one_hot) + list(target_enc)
    model.fit(train[cols], train[DELAY_TARGET])
    proba = model.predict_proba(test[cols])[:, 1]
    return {"feature set": name,
            "roc_auc": roc_auc_score(test[DELAY_TARGET], proba),
            "accuracy": accuracy_score(test[DELAY_TARGET], proba >= 0.5)}

ITEM = ["Product Price", "Order Item Quantity", "Order Item Discount Rate"]
GEO = ["Order City", "Order Country", "Order State", "Customer City", "Customer State"]

experiments = [
    ("Shipping mode only", [], ["Shipping Mode"]),
    ("Base (Phase 0)", DELAY_NUMERIC, DELAY_CATEGORICAL),
    ("Base + item price/quantity/discount", DELAY_NUMERIC + ITEM, DELAY_CATEGORICAL),
    ("Base + store latitude/longitude", DELAY_NUMERIC + ["Latitude", "Longitude"], DELAY_CATEGORICAL),
    ("Base + 5 location columns (target enc.)", DELAY_NUMERIC, DELAY_CATEGORICAL, GEO),
    ("Base + product name (target enc.)", DELAY_NUMERIC, DELAY_CATEGORICAL, ["Product Name"]),
    ("Base + order city & country (target enc.)", DELAY_NUMERIC, DELAY_CATEGORICAL, ["Order City", "Order Country"]),
]

ablation = pd.DataFrame([try_features(*e) for e in experiments])
ablation.round(4)

,feature set,roc_auc,accuracy
0,Shipping mode only,0.7260,0.6925
1,Base (Phase 0),0.7620,0.6970
2,Base + item price/quantity/discount,0.7645,0.7064
3,Base + store latitude/longitude,0.7611,0.7017
4,Base + 5 location columns (target enc.),0.7699,0.7064
5,Base + product name (target enc.),0.7661,0.7085
6,Base + order city & country (target enc.),0.7695,0.7052


In [3]:
variants = {
    "XGBoost v1 (base features)": (DELAY_FEATURES, delay_preprocessor),
    "XGBoost v2 (+ city/country encoding)": (DELAY_FEATURES_V2, delay_preprocessor_v2),
}

folds = []
for month, tr, te in rolling_origin_months(data, "2017-04", "2017-09"):
    train_m, test_m = data.loc[tr], data.loc[te]
    out = test_m[["Order Id", "period", DELAY_TARGET]].copy()
    for name, (feats, prep) in variants.items():
        model = Pipeline([("prep", prep()), ("model", xgb())])
        model.fit(train_m[feats], train_m[DELAY_TARGET])
        out[name] = model.predict_proba(test_m[feats])[:, 1]
    folds.append(out)
    print(month, "done")

cv = pd.concat(folds, ignore_index=True)
y_cv = cv[DELAY_TARGET]

rows = []
for m in variants:
    auc, lo, hi = cluster_bootstrap(y_cv, cv[m], cv["Order Id"], roc_auc_score, n_boot=1000)
    rows.append({"model": m, "accuracy": accuracy_score(y_cv, cv[m] >= 0.5),
                 "roc_auc": auc, "auc_CI": f"[{lo:.3f}, {hi:.3f}]"})
print(pd.DataFrame(rows).round(3).to_string(index=False))

v1, v2 = list(variants)
d, lo, hi = cluster_bootstrap(y_cv, cv[v2], cv["Order Id"], roc_auc_score, pred_b=cv[v1], n_boot=1000)
print(f"\nAUC gain v2 - v1: {d:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

2017-04 done
2017-05 done
2017-06 done
2017-07 done
2017-08 done
2017-09 done
                               model  accuracy  roc_auc         auc_CI
          XGBoost v1 (base features)     0.707    0.763 [0.753, 0.772]
XGBoost v2 (+ city/country encoding)     0.704    0.768 [0.759, 0.778]

AUC gain v2 - v1: 0.0057  95% CI [-0.0013, 0.0127]
